# 03 - Business & Underwriting View

This notebook translates multistate survival model outputs into quantities
relevant for insurance underwriting in the DACH region:

1. **Cost bucketing** - Inpatient, outpatient, pharmacy, rehabilitation, long-term care
2. **Expected time in high-cost state** - Mean sojourn time in expensive disease states
3. **Underwriting quantities** - Excess mortality, expected claim cost, risk classification
4. **DACH cost reference tables** - German DRG, Swiss TARMED, Austrian LKF benchmarks

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pathlib
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.1)

DATA_DIR = pathlib.Path("../data/synthea")
OUTPUT_DIR = pathlib.Path("../outputs/business")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Cost bucketing ──────────────────────────────────────────────────────────
# Load encounter and claims data
encounters = pd.read_csv(DATA_DIR / "encounters.csv", parse_dates=["START", "STOP"])
medications = pd.read_csv(DATA_DIR / "medications.csv", parse_dates=["START", "STOP"])

# Map encounter classes to cost buckets
COST_BUCKET_MAP = {
    "inpatient": "Inpatient",
    "emergency": "Inpatient",
    "urgentcare": "Inpatient",
    "ambulatory": "Outpatient",
    "outpatient": "Outpatient",
    "wellness": "Outpatient",
    "snf": "Long-term care",
    "home": "Long-term care",
    "hospice": "Long-term care",
}

encounters["cost_bucket"] = (
    encounters["ENCOUNTERCLASS"]
    .str.lower()
    .map(COST_BUCKET_MAP)
    .fillna("Other")
)

# Cost summary by bucket
cost_cols = ["TOTAL_CLAIM_COST", "PAYER_COVERAGE"]
for col in cost_cols:
    if col not in encounters.columns:
        encounters[col] = np.random.lognormal(7, 1.5, len(encounters))  # synthetic fallback

encounters["patient_cost"] = encounters["TOTAL_CLAIM_COST"] - encounters["PAYER_COVERAGE"]

bucket_summary = (
    encounters.groupby("cost_bucket")
    .agg(
        n_encounters=("Id", "count"),
        total_cost=("TOTAL_CLAIM_COST", "sum"),
        mean_cost=("TOTAL_CLAIM_COST", "mean"),
        median_cost=("TOTAL_CLAIM_COST", "median"),
        p90_cost=("TOTAL_CLAIM_COST", lambda x: x.quantile(0.9)),
    )
    .sort_values("total_cost", ascending=False)
)

print("Cost bucket summary:")
print(bucket_summary.to_string())

# Add pharmacy costs
if "TOTALCOST" in medications.columns:
    pharmacy_total = medications["TOTALCOST"].sum()
else:
    pharmacy_total = len(medications) * 150  # estimate

print(f"\nTotal pharmacy cost (estimated): EUR {pharmacy_total:,.0f}")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bucket_summary["total_cost"].plot.bar(ax=axes[0], color="#4C72B0")
axes[0].set_title("Total cost by bucket")
axes[0].set_ylabel("Total cost (USD)")
axes[0].tick_params(axis="x", rotation=45)

bucket_summary["mean_cost"].plot.bar(ax=axes[1], color="#55A868")
axes[1].set_title("Mean cost per encounter by bucket")
axes[1].set_ylabel("Mean cost (USD)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ── Expected time in high-cost state ────────────────────────────────────────
# Define high-cost states from the multistate models
# For CVD: Post-event and HF states are high-cost
# For Diabetes: ESRD and Macrovascular are high-cost

# Simulate sojourn times from a simple Markov model
# (In production, these come from the trained multistate models)

# CVD transition intensity estimates (per year)
cvd_q_matrix = np.array([
    #  RF     CHD    Post   HF    Death
    [-0.08,  0.05,  0.00,  0.00, 0.03],   # Risk Factors
    [ 0.00, -0.12,  0.08,  0.00, 0.04],   # Stable CHD
    [ 0.00,  0.00, -0.15,  0.10, 0.05],   # Post-event
    [ 0.00,  0.00,  0.00, -0.20, 0.20],   # HF
    [ 0.00,  0.00,  0.00,  0.00, 0.00],   # Death (absorbing)
])

# Mean sojourn time = -1/q_ii
cvd_states = ["Risk Factors", "Stable CHD", "Post-event", "HF", "Death"]
cvd_sojourn = pd.DataFrame({
    "State": cvd_states[:-1],
    "Mean sojourn (years)": [-1.0 / cvd_q_matrix[i, i] for i in range(4)],
    "High-cost": [False, False, True, True],
})

# Diabetes transition intensities
dm_q_matrix = np.array([
    #  Pre    T2D    Micro  Macro  ESRD   Death
    [-0.06,  0.05,  0.00,  0.00,  0.00,  0.01],   # Prediabetes
    [ 0.00, -0.10,  0.06,  0.02,  0.00,  0.02],   # T2D
    [ 0.00,  0.00, -0.08,  0.04,  0.01,  0.03],   # Microvascular
    [ 0.00,  0.00,  0.00, -0.12,  0.05,  0.07],   # Macrovascular
    [ 0.00,  0.00,  0.00,  0.00, -0.25,  0.25],   # ESRD
    [ 0.00,  0.00,  0.00,  0.00,  0.00,  0.00],   # Death
])

dm_states = ["Prediabetes", "T2D", "Microvascular", "Macrovascular", "ESRD", "Death"]
dm_sojourn = pd.DataFrame({
    "State": dm_states[:-1],
    "Mean sojourn (years)": [-1.0 / dm_q_matrix[i, i] for i in range(5)],
    "High-cost": [False, False, False, True, True],
})

print("CVD: Mean sojourn times")
print(cvd_sojourn.to_string(index=False))
print(f"\nExpected time in high-cost CVD state: "
      f"{cvd_sojourn[cvd_sojourn['High-cost']]['Mean sojourn (years)'].sum():.1f} years")

print("\nDiabetes: Mean sojourn times")
print(dm_sojourn.to_string(index=False))
print(f"\nExpected time in high-cost DM state: "
      f"{dm_sojourn[dm_sojourn['High-cost']]['Mean sojourn (years)'].sum():.1f} years")

In [ ]:
# ── Translate model outputs to underwriting quantities ──────────────────────

# Excess mortality ratio (EMR) by disease state
# Reference: general population mortality ~ 0.01 per year for age 50-60
baseline_mortality = 0.01  # per year, rough average

emr_table = pd.DataFrame({
    "State": ["Risk Factors only", "Stable CHD", "Post-MI/Stroke", "Heart Failure (NYHA II-III)",
              "Prediabetes", "T2D (controlled)", "T2D + Microvascular",
              "T2D + Macrovascular", "ESRD / Dialysis"],
    "Estimated annual mortality": [0.015, 0.04, 0.05, 0.20,
                                    0.012, 0.02, 0.03, 0.07, 0.25],
    "EMR (%)": [None] * 9,  # computed below
    "Suggested risk class": [
        "Standard", "Substandard +50", "Substandard +100", "Decline / rated",
        "Standard", "Standard / +25", "Substandard +50", "Substandard +150", "Decline",
    ],
})
emr_table["EMR (%)"] = (
    (emr_table["Estimated annual mortality"] / baseline_mortality - 1) * 100
).round(0).astype(int)

print("\nExcess Mortality Ratio (EMR) table for underwriting:")
print(emr_table.to_string(index=False))

# Expected annual claim cost by state (EUR)
cost_table = pd.DataFrame({
    "State": ["Healthy / Risk Factors", "Stable CHD", "Post-MI (year 1)",
              "Post-MI (year 2+)", "Heart Failure",
              "T2D (controlled)", "T2D + Retinopathy", "T2D + CKD Stage 3",
              "ESRD / Dialysis"],
    "Annual cost EUR (est.)": [
        2_500, 5_500, 35_000, 8_000, 18_000,
        4_500, 7_000, 12_000, 55_000,
    ],
    "Cost bucket": [
        "Outpatient + Pharmacy", "Outpatient + Pharmacy", "Inpatient + Rehab",
        "Outpatient + Pharmacy", "Inpatient + Long-term",
        "Outpatient + Pharmacy", "Outpatient + Specialist", "Outpatient + Specialist",
        "Inpatient (Dialysis)",
    ],
})

print("\nExpected annual claim cost by state:")
print(cost_table.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(cost_table["State"], cost_table["Annual cost EUR (est.)"], color="#4C72B0")
ax.set_xlabel("Annual cost (EUR)")
ax.set_title("Estimated annual claim cost by disease state")
ax.invert_yaxis()
for i, v in enumerate(cost_table["Annual cost EUR (est.)"]):
    ax.text(v + 500, i, f"EUR {v:,.0f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── DACH cost reference tables ──────────────────────────────────────────────

# German DRG reference costs (InEK, approximate 2023 figures)
de_drg = pd.DataFrame({
    "DRG": ["F49A", "F49B", "F62A", "F62B", "F71B", "L71A", "L71B"],
    "Description": [
        "Invasive cardiac diagnostics (with complex intervention)",
        "Invasive cardiac diagnostics (without complex intervention)",
        "Heart failure with complex diagnostics",
        "Heart failure without complex diagnostics",
        "Non-surgical cardiac conditions, >2 days",
        "Dialysis with complex comorbidities",
        "Dialysis without complex comorbidities",
    ],
    "Mean cost EUR": [8_200, 3_800, 5_600, 3_200, 2_900, 12_500, 7_800],
    "Mean LOS days": [5.2, 2.8, 8.1, 5.5, 4.2, 9.5, 6.3],
})

# Swiss reference (TARMED + SwissDRG, approximate)
ch_ref = pd.DataFrame({
    "Category": [
        "Acute MI (primary PCI)", "Stroke (thrombolysis)",
        "Heart failure hospitalisation", "Dialysis (per session)",
        "Diabetes outpatient (quarterly)",
    ],
    "Mean cost CHF": [28_000, 22_000, 12_000, 580, 450],
})

# Austrian LKF reference
at_ref = pd.DataFrame({
    "LKF Group": ["HDG05.01", "HDG05.03", "HDG11.02"],
    "Description": [
        "Acute coronary syndrome",
        "Heart failure (inpatient)",
        "Renal replacement therapy",
    ],
    "Mean cost EUR": [15_000, 6_500, 45_000],
})

print("=" * 70)
print("DACH COST REFERENCE TABLES")
print("=" * 70)
print("\n--- Germany (InEK DRG 2023) ---")
print(de_drg.to_string(index=False))
print("\n--- Switzerland (SwissDRG / TARMED) ---")
print(ch_ref.to_string(index=False))
print("\n--- Austria (LKF) ---")
print(at_ref.to_string(index=False))

# Save reference tables
with pd.ExcelWriter(OUTPUT_DIR / "dach_cost_reference.xlsx") as writer:
    de_drg.to_excel(writer, sheet_name="Germany_DRG", index=False)
    ch_ref.to_excel(writer, sheet_name="Switzerland", index=False)
    at_ref.to_excel(writer, sheet_name="Austria_LKF", index=False)
    cost_table.to_excel(writer, sheet_name="State_costs", index=False)
    emr_table.to_excel(writer, sheet_name="EMR_table", index=False)

print(f"\nReference tables saved to {OUTPUT_DIR / 'dach_cost_reference.xlsx'}")